# 01. GroupBy Basics: Split-Apply-Combine

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week10/01.GroupBy-Basics/notebooks/01_01.GroupBy-Basics.ipynb)

## Overview
In data science, we frequently need to divide datasets into meaningful categories (such as campuses, courses, or departments), compute summary statistics on each subset, and assemble the results into a clean overview.

This workflow is formalised as the **Split-Apply-Combine** paradigm:
1. **Split**: Break the data into groups based on key columns.
2. **Apply**: Compute summary statistics (mean, sum, count, min, max) on each group independently.
3. **Combine**: Assemble the computed group metrics back into a unified DataFrame or Series.

### Learning Objectives
- Instantiate and inspect `DataFrameGroupBy` objects.
- Extract single cohorts using `.get_group()` and count group observations with `.size()`.
- Aggregate single vs. multiple numeric columns.
- Control output indexing using `as_index=False`.

## 1. Setup: Australian Catholic University (ACU) Student Cohorts

We create a realistic student dataset featuring ACU campuses, courses, Weighted Average Marks (WAM), attendance, and completed units.

In [ ]:
import pandas as pd
import numpy as np

# Create student cohort dataset
df_students = pd.DataFrame({
    'student_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'name': ['Liam Nguyen', 'Emma Watson', 'Oliver Brown', 'Sophia Vu', 'Noah Taylor', 'Ava Wilson', 'Lucas Martin', 'Mia Anderson'],
    'campus': ['North Sydney', 'Melbourne', 'North Sydney', 'Brisbane', 'Melbourne', 'Strathfield', 'Brisbane', 'Strathfield'],
    'study_mode': ['Full-time', 'Part-time', 'Full-time', 'Full-time', 'Part-time', 'Full-time', 'Part-time', 'Full-time'],
    'course': ['Bachelor of IT', 'Bachelor of Nursing', 'Bachelor of IT', 'Bachelor of Business', 'Bachelor of IT', 'Bachelor of Education', 'Bachelor of Business', 'Bachelor of Education'],
    'wam': [82.5, 76.0, 89.0, 71.5, 84.0, 78.5, 68.0, 91.0],
    'attendance': [92, 85, 95, 78, 88, 82, 70, 96],
    'units_completed': [8, 4, 12, 6, 8, 10, 4, 12]
})

print("ACU Student Cohort Data:")
display(df_students)

## 2. The Split Step: Inspecting GroupBy Objects

Calling `df.groupby('campus')` creates a lazy `DataFrameGroupBy` object without computing statistics yet.
We can inspect:
- `.ngroups`: Number of distinct groups.
- `.groups`: Dictionary mapping group names to original row indices.
- `.size()`: Number of records in each group.
- `.get_group('Name')`: Extracts all rows belonging to a specific group.

In [ ]:
grouped = df_students.groupby('campus')

print(f"Total groups: {grouped.ngroups}")
print(f"Group labels: {list(grouped.groups.keys())}")
print("\nCohort Counts (.size()):")
display(grouped.size())

print("\nExtracting 'Melbourne' students with .get_group():")
display(grouped.get_group('Melbourne'))

## 3. The Apply & Combine Steps: Computing Aggregations

When you invoke a statistical method like `.mean()` or `.sum()`:
1. **Single column selection** `['wam']` returns a Pandas `Series`.
2. **Multiple column selection** `[['wam', 'attendance']]` returns a Pandas `DataFrame`.

In [ ]:
# Aggregating a single column -> Series
avg_wam = df_students.groupby('campus')['wam'].mean()
print("Average WAM by Campus (Series):")
display(avg_wam.round(2))

# Aggregating multiple columns -> DataFrame
multi_stats = df_students.groupby('campus')[['wam', 'attendance']].mean()
print("\nAverage WAM and Attendance by Campus (DataFrame):")
display(multi_stats.round(2))

## 4. Retaining Columns with `as_index=False`

By default, `df.groupby()` sets the grouping keys as the row `Index` of the resulting DataFrame.
If you want to keep the grouping key as a standard tabular column, set `as_index=False`.

In [ ]:
# Grouping with as_index=False
tabular_summary = df_students.groupby('campus', as_index=False)['wam'].mean()
print("Summary with as_index=False:")
display(tabular_summary)
print(f"Columns in result: {tabular_summary.columns.tolist()}")

## 5. Practical Exercises

### Exercise 1: Emergency Department Wait Times
Using the Australian hospital admissions dataset below:
1. Group the admissions by `department`.
2. Calculate the average `wait_time_mins` for each department.

In [ ]:
hospitals = pd.DataFrame({
    'patient_id': [201, 202, 203, 204, 205, 206],
    'hospital': ['Sydney', 'Melbourne', 'Sydney', 'Brisbane', 'Melbourne', 'Brisbane'],
    'department': ['Emergency', 'Cardiology', 'Cardiology', 'Emergency', 'Pediatrics', 'Emergency'],
    'wait_time_mins': [45, 15, 20, 50, 10, 60],
    'length_of_stay_days': [2, 5, 4, 1, 3, 2]
})

# --- Student Code Here ---
# dept_wait = ...

# --- Solution ---
dept_wait = hospitals.groupby('department')['wait_time_mins'].mean()
display(dept_wait.round(1))

### Exercise 2: Maximum Length of Stay with `as_index=False`
Group the `hospitals` dataset by `hospital` with `as_index=False`, and compute the maximum `length_of_stay_days`.

In [ ]:
# --- Student Code Here ---
# max_stay = ...

# --- Solution ---
max_stay = hospitals.groupby('hospital', as_index=False)['length_of_stay_days'].max()
display(max_stay)

## 6. Key Takeaways

1. **Split-Apply-Combine**: The engine behind all SQL `GROUP BY` and Pandas aggregation.
2. **`grouped.size()`**: Quickly checks how many observations exist in each cohort.
3. **Single vs Double Brackets**: `['col']` gives a Series, `[['col']]` preserves a DataFrame.
4. **`as_index=False`**: Prevents the grouping key from becoming the row index, keeping it as a regular column for downstream merges.